# Testing AE convnext

In [8]:
import torch
import torch.nn as nn
from models.blocks import ConvNeXtcausal
from torch.nn.utils.parametrizations import weight_norm
from transformers import EncodecModel

In [2]:
class EncoderFast(nn.Module):
    def __init__(self, in_channels: int, dim: int, latent_dim: int, inter_channels: int, num_blocks: int):
        super(EncoderFast, self).__init__()

        stride = [4, 4, 4]
        self.conv1 = weight_norm(nn.Conv1d(in_channels, dim, kernel_size=7, padding=6))
        self.blocks = [ConvNeXtcausal(dim, inter_channels) for _ in range(num_blocks)]
        self.stages= nn.ModuleList()

        for s in stride:
            stage = nn.Sequential(
                *self.blocks,
                nn.Conv1d(dim, dim, kernel_size=s, stride=s, padding=s-1)
            
            )
            self.stages.append(stage)

        self.norm = nn.LayerNorm(dim, eps=1e-6)
        self.proj = nn.Linear(dim, latent_dim)
    
    def forward(self, x):
        x = self.conv1(x)
        print(x.shape)
        for stage in self.stages:
            x = stage(x)
        print(x.shape)
        #x = x.mean(dim=-1)
        x = x.transpose(1,2)
        print(x.shape)
        x = self.norm(x)
        print(x.shape)
        x = self.proj(x)
        x = x.transpose(1,2)
        return x

In [3]:
x = torch.randn(1, 1, 24000)
dim = x.size(-1)
encoder = EncoderFast(in_channels=1, dim=dim//4, latent_dim=dim//8, inter_channels=128, num_blocks=2)
out = encoder(x)
print(f"Output shape: {out.shape}")

torch.Size([1, 6000, 24006])
torch.Size([1, 6000, 377])
torch.Size([1, 377, 6000])
torch.Size([1, 377, 6000])
Output shape: torch.Size([1, 3000, 377])


In [12]:
x = torch.randn(1, 2, 48000)
model = EncodecModel.from_pretrained('facebook/encodec_48khz')
with torch.no_grad():
    emb = model.encoder(x)
print(f"Encoder output shape: {emb.shape}")

Loading weights: 100%|██████████| 224/224 [00:00<00:00, 8872.39it/s]

Encoder output shape: torch.Size([1, 128, 150])


In [13]:
class Decoder(nn.Module):
    def __init__(self, in_channels: int, dim: int, shift_dim: int, inter_channels: int, num_blocks: int):
        super(Decoder, self).__init__()
        self.conv = nn.Conv1d(in_channels, dim, kernel_size=7, padding=3)
        self.norm = nn.LayerNorm(dim, eps=1e-6)
        self.blocks = nn.ModuleList([ConvNeXtcausal(dim, inter_channels) for _ in range(num_blocks)])
        self.linear1 = nn.Linear(dim, dim)
        self.linear2 = nn.Linear(dim, shift_dim, bias=False) 
        # (B, shift_dim, T) -> (B, 1 , shift_dim * T)
    
    def forward(self, x):
        x = self.conv(x)
        print(x.shape)
        x = x.transpose(1, 2)  # (B, T, dim)
        print(x.shape)
        x = self.norm(x)
        x = x.transpose(1, 2)  # (B, dim, T)
        print(x.shape)
        for block in self.blocks:
            x = block(x)
        print(x.shape)
        x = x.transpose(1, 2)  # (B, T, dim)
        print(x.shape)
        x = self.linear1(x)
        print(x.shape)
        x = self.linear2(x) # (B, T, shift_dim)
        print(x.shape)
        x = x.view(x.size(0), -1) # (B, shift_dim * T)

        return x

In [14]:
s_dim = x.size(-1) // emb.size(-1)
decoder = Decoder(in_channels=128, dim=512, shift_dim=s_dim, inter_channels=256, num_blocks=2)
y = decoder(emb)
print(f"Decoder output shape: {y.shape}")
s_dim

torch.Size([1, 512, 150])
torch.Size([1, 150, 512])
torch.Size([1, 512, 150])
torch.Size([1, 512, 150])
torch.Size([1, 150, 512])
torch.Size([1, 150, 512])
torch.Size([1, 150, 320])
Decoder output shape: torch.Size([1, 48000])


320